# Semantic-group robustness experiment

This notebook creates a leakage-resistant grouped split and evaluates Linear SVM, the fine-tuned encoder, and their WAR policies. It uses no manual or LLM annotation.

In [ ]:
%pip install -q "sentence-transformers>=2.6,<4" "scikit-learn>=1.3"

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '0'
os.environ['TRANSFORMERS_OFFLINE'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

work = Path('/kaggle/working')
repo = work / 'agent-routing-recsys-artifact'
if repo.exists():
    shutil.rmtree(repo)

input_root = Path('/kaggle/input')
script_candidates = list(input_root.rglob('scripts/create_semantic_group_split.py'))
if script_candidates:
    # Kaggle normally extracts uploaded ZIP datasets automatically.
    source_repo = script_candidates[0].parents[1]
    print('Using extracted Kaggle input:', source_repo)
    shutil.copytree(source_repo, repo)
else:
    zip_candidates = list(input_root.rglob('agent-routing-recsys-artifact-kaggle.zip'))
    if not zip_candidates:
        zip_candidates = list(input_root.rglob('*.zip'))
    if not zip_candidates:
        visible = [str(path) for path in input_root.iterdir()]
        raise FileNotFoundError(
            'Artifact input not found. Attach the uploaded Kaggle dataset. '
            f'Visible inputs: {visible}'
        )
    print('Using ZIP:', zip_candidates[0])
    shutil.unpack_archive(str(zip_candidates[0]), str(work))
    if not repo.exists():
        matches = [p.parents[1] for p in work.rglob('scripts/create_semantic_group_split.py')]
        if not matches:
            raise FileNotFoundError('Could not locate the extracted artifact root.')
        repo = matches[0]
print('Repository:', repo)
print('GPU available:')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
processed = repo / 'data' / 'processed'
split_out = work / 'semantic_group_split'
cmd = [
    sys.executable, str(repo / 'scripts' / 'create_semantic_group_split.py'),
    '--input_csvs', ','.join(str(processed / name) for name in [
        'wildchat_agent12_balanced_3000_equalized_train.csv',
        'wildchat_agent12_balanced_3000_equalized_dev.csv',
        'wildchat_agent12_balanced_3000_equalized_test.csv',
    ]),
    '--embedding_model', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    '--similarity_threshold', '0.82',
    '--assignment_trials', '20000',
    '--seed', '42',
    '--output_dir', str(split_out),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=repo)

In [ ]:
results_out = work / 'semantic_group_robustness'
cmd = [
    sys.executable, str(repo / 'scripts' / 'evaluate_semantic_group_robustness.py'),
    '--train_csv', str(split_out / 'semantic_group_train.csv'),
    '--dev_csv', str(split_out / 'semantic_group_dev.csv'),
    '--test_csv', str(split_out / 'semantic_group_test.csv'),
    '--split_manifest', str(split_out / 'semantic_group_split_manifest.json'),
    '--seeds', '42,43,44',
    '--thresholds', '0.3,0.4,0.5,0.6,0.7,0.8,0.9',
    '--war_lambdas', '0,0.02,0.05,0.1,0.15',
    '--war_cost_tiers', str(repo / 'config' / 'agent_cost_tiers_12.json'),
    '--embedder_model', 'sentence-transformers/all-mpnet-base-v2',
    '--encoder_model_name', 'sentence-transformers/all-mpnet-base-v2',
    '--encoder_epochs', '3',
    '--encoder_batch_size', '8',
    '--encoder_device', 'cuda',
    '--output_dir', str(results_out),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=repo)

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

display(pd.read_csv(results_out / 'semantic_group_test_aggregate.csv'))
display(Markdown((results_out / 'semantic_group_robustness_summary.md').read_text()))
manifest = json.loads((split_out / 'semantic_group_split_manifest.json').read_text())
print(json.dumps({
    'actual_rows': manifest['actual_rows'],
    'num_semantic_groups': manifest['num_semantic_groups'],
    'largest_group_rows': manifest['largest_group_rows'],
    'max_cross_split_cosine': manifest['max_cross_split_cosine'],
    'max_cross_split_below_threshold': manifest['max_cross_split_below_threshold'],
}, indent=2))

In [ ]:
bundle = work / 'semantic_group_robustness_outputs'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(split_out, bundle / 'split')
shutil.copytree(results_out, bundle / 'results')
archive = shutil.make_archive(str(work / 'semantic_group_robustness_outputs'), 'zip', bundle)
print('Download:', archive)